# Model Modality Tests
Tests different capabilities of a deployed LLM endpoint: chat, reasoning, vision, embeddings, reranking, streaming, and structured output.

Set the `BASE_URL` below to the endpoint you want to test.

In [ ]:
from bellmira.llm_model.llm_model_client import ModelClient
import base64, json, requests

BASE_URL = "http://localhost:8080/"  # <-- change this

model = ModelClient(base_url=BASE_URL)
MODEL_NAME = model.get_model_name()
print(f"Model: {MODEL_NAME}")

## 1. Basic Chat

In [ ]:
req = model.build_chat_request(
    user_prompt="What is the capital of Portugal?",
    system_prompt="You are a helpful assistant.",
    model_name=MODEL_NAME,
    temperature=0.0,
)
response = model.send_request(req)

if response.status_code == 200:
    print("PASS - Basic Chat")
    print(response.json()["choices"][0]["message"]["content"])
else:
    print(f"FAIL - Status {response.status_code}: {response.text}")

## 2. Reasoning
Tests chain-of-thought / thinking mode. Expects the model to reason step-by-step before answering.

In [ ]:
req = model.build_chat_request(
    user_prompt=(
        "A train leaves station A at 08:00 travelling at 80 km/h. "
        "Another train leaves station B (320 km away) at 09:00 travelling at 120 km/h towards A. "
        "At what time do they meet?"
    ),
    system_prompt="Think step by step before giving your final answer.",
    model_name=MODEL_NAME,
    temperature=0.0,
    enable_thinking=True,
)
response = model.send_request(req)

if response.status_code == 200:
    msg = response.json()["choices"][0]["message"]
    reasoning = msg.get("reasoning_content") or "(no separate reasoning field)"
    answer = msg.get("content", "")
    print("PASS - Reasoning")
    print("--- Reasoning ---")
    print(reasoning[:500], "..." if len(reasoning) > 500 else "")
    print("--- Answer ---")
    print(answer)
else:
    print(f"FAIL - Status {response.status_code}: {response.text}")

## 3. Vision
Tests image understanding by sending a base64-encoded image with a question.
Replace `IMAGE_PATH` with a local image file, or provide a URL-fetched image.

In [ ]:
IMAGE_PATH = None  # e.g. "/path/to/image.png" — set to None to skip

if IMAGE_PATH is None:
    # Generate a minimal 1x1 white PNG in memory for a smoke test
    import struct, zlib
    def _minimal_png():
        sig = b'\x89PNG\r\n\x1a\n'
        def chunk(name, data): crc = zlib.crc32(name + data) & 0xffffffff; return struct.pack('>I', len(data)) + name + data + struct.pack('>I', crc)
        ihdr = chunk(b'IHDR', struct.pack('>IIBBBBB', 1, 1, 8, 2, 0, 0, 0))
        idat = chunk(b'IDAT', zlib.compress(b'\x00\xff\xff\xff'))
        iend = chunk(b'IEND', b'')
        return sig + ihdr + idat + iend
    image_b64 = base64.b64encode(_minimal_png()).decode()
else:
    with open(IMAGE_PATH, "rb") as f:
        image_b64 = base64.b64encode(f.read()).decode()

req = model.build_chat_request(
    user_prompt="Describe what you see in this image.",
    model_name=MODEL_NAME,
    temperature=0.0,
    image_prompt=image_b64,
)
response = model.send_request(req)

if response.status_code == 200:
    print("PASS - Vision")
    print(response.json()["choices"][0]["message"]["content"])
else:
    print(f"FAIL - Status {response.status_code}: {response.text}")

## 4. Embeddings
Tests that the endpoint returns a vector for a given input text.

In [ ]:
req = model.build_embedding_request(
    input_text="The quick brown fox jumps over the lazy dog.",
    model_name=MODEL_NAME,
)
response = model.send_request(req)

if response.status_code == 200:
    embedding = response.json()["data"][0]["embedding"]
    print("PASS - Embeddings")
    print(f"Embedding dimension: {len(embedding)}")
    print(f"First 5 values: {embedding[:5]}")
else:
    print(f"FAIL - Status {response.status_code}: {response.text}")

## 5. Reranking
Tests that the endpoint ranks documents by relevance to a query.

In [ ]:
query = "What is a neural network?"
documents = [
    "A neural network is a series of algorithms that attempt to recognize relationships in data.",
    "The Eiffel Tower is located in Paris, France.",
    "Deep learning uses multiple layers of neural networks to learn representations.",
]

req = model.build_rerank_request(
    query=query,
    documents=documents,
    modelname=MODEL_NAME,
)
response = model.send_request(req)

if response.status_code == 200:
    results = response.json()["results"]
    print("PASS - Reranking")
    for r in results:
        print(f"  [{r['relevance_score']:.4f}] {r['document']['text'][:60]}")
else:
    print(f"FAIL - Status {response.status_code}: {response.text}")

## 6. Streaming
Tests that the endpoint can stream tokens back incrementally.

In [ ]:
req = model.build_chat_request(
    user_prompt="Count from 1 to 10, one number per line.",
    model_name=MODEL_NAME,
    temperature=0.0,
)

try:
    tokens = []
    for token in model.stream_chat_response(req):
        print(token, end="", flush=True)
        tokens.append(token)
    print()
    print(f"PASS - Streaming ({len(tokens)} chunks)")
except Exception as e:
    print(f"FAIL - Streaming: {e}")

## 7. Structured Output (JSON Schema)
Tests that the model can produce output conforming to a given JSON schema.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "capital": {"type": "string"},
        "population_millions": {"type": "number"}
    },
    "required": ["name", "capital", "population_millions"]
}

req = model.build_chat_request(
    user_prompt="Give me facts about Portugal as JSON.",
    model_name=MODEL_NAME,
    temperature=0.0,
    json_schema=schema,
)
response = model.send_request(req)

if response.status_code == 200:
    content = response.json()["choices"][0]["message"]["content"]
    try:
        parsed = json.loads(content)
        print("PASS - Structured Output")
        print(json.dumps(parsed, indent=2))
    except json.JSONDecodeError:
        print(f"FAIL - Response not valid JSON:\n{content}")
else:
    print(f"FAIL - Status {response.status_code}: {response.text}")

## Summary
Re-run cells above individually. A `FAIL` on vision/reranking/embeddings usually means the endpoint does not support that modality — that is expected for chat-only or embedding-only models.